### Clone WasteLens Repository

In [1]:
import subprocess
from pathlib import Path

REPO = Path("/content/WasteLens")

if not REPO.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/Anik00223/WasteLens.git",
         str(REPO)],
        check=True
    )

print("WasteLens repository is ready!")

WasteLens repository is ready!


###Verify OOD Model

In [2]:
import json
from pathlib import Path

REPO = Path("/content/WasteLens")
MODEL_DIR = REPO / "web" / "model"

# 1. Load model configuration
model_path = MODEL_DIR / "model.json"
model_data = json.loads(model_path.read_text())

# 2. Check classification and rejection layers
layers = model_data["modelTopology"]["config"]["layers"]

layer_names = {
    layer.get("config", {}).get("name")
    for layer in layers
}

assert "predictions" in layer_names, "Classification output missing"
assert "reject" in layer_names, "OOD rejection output missing"

print("Classification head: FOUND")
print("OOD rejection head: FOUND")

# 3. Verify class labels
labels_path = MODEL_DIR / "labels.json"
labels = json.loads(labels_path.read_text())

print("\nWaste categories:", labels)

# 4. Verify model weight files
manifest = model_data["weightsManifest"]

for group in manifest:
    for filename in group["paths"]:
        weight_path = MODEL_DIR / filename
        assert weight_path.is_file(), f"Missing weight file: {filename}"
        print("Weight file FOUND:", filename)

print("\nSUCCESS: Model files verified!")

Classification head: FOUND
OOD rejection head: FOUND

Waste categories: ['recyclable', 'organic', 'hazardous', 'general trash']
Weight file FOUND: group1-shard1of3.bin
Weight file FOUND: group1-shard2of3.bin
Weight file FOUND: group1-shard3of3.bin

SUCCESS: Model files verified!


###Start Static Web Server

In [4]:
import subprocess
import sys
import time
import socket
import requests
from pathlib import Path

# 1. Set the website directory
REPO = Path("/content/WasteLens")
WEB_DIR = REPO / "web"

assert WEB_DIR.is_dir(), "WasteLens web directory not found!"

# 2. Stop our previous server if this cell is being rerun
old_server = globals().get("server_process")

if old_server is not None and old_server.poll() is None:
    print("Stopping previous WasteLens server...")
    old_server.terminate()
    old_server.wait(timeout=10)

# 3. Select an available port
PORT = 8000

def port_is_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("0.0.0.0", port))
            return True
        except OSError:
            return False

if not port_is_free(PORT):
    PORT = 8001

if not port_is_free(PORT):
    raise RuntimeError("Ports 8000 and 8001 are occupied. Restart the runtime.")

# 4. Start the static web server
server_log_file = open("/content/wastelens_server.log", "w")

server_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "http.server",
        str(PORT),
        "--bind", "0.0.0.0",
        "--directory", str(WEB_DIR)
    ],
    stdout=server_log_file,
    stderr=subprocess.STDOUT
)

# 5. Verify that the website is accessible
for attempt in range(30):
    if server_process.poll() is not None:
        raise RuntimeError(
            "Web server stopped unexpectedly:\n" +
            Path("/content/wastelens_server.log").read_text()
        )

    try:
        response = requests.get(
            f"http://127.0.0.1:{PORT}/",
            timeout=3
        )

        if response.status_code == 200:
            print("Website loaded successfully!")
            print("HTTP Status:", response.status_code)
            break

    except requests.RequestException:
        pass

    time.sleep(1)

else:
    raise RuntimeError("Website failed to start. Check server logs.")

# 6. Verify the TensorFlow.js model is accessible
model_response = requests.get(
    f"http://127.0.0.1:{PORT}/model/model.json",
    timeout=10
)

assert model_response.status_code == 200

print("Model JSON: ACCESSIBLE")
print("Server running on port:", PORT)
print("WASTELENS SERVER READY!")

Stopping previous WasteLens server...
Website loaded successfully!
HTTP Status: 200
Model JSON: ACCESSIBLE
Server running on port: 8000
WASTELENS SERVER READY!


###Open WasteLens Demo

In [6]:
from google.colab import output
from IPython.display import HTML, display

# Check whether the server is running
assert server_process.poll() is None, "Server is not running!"

# Generate a fresh Colab link
demo_url = output.eval_js(
    f"google.colab.kernel.proxyPort({PORT})"
).rstrip("/")

print("Your WasteLens Demo:")
print(demo_url)

display(HTML(
    f'<a href="{demo_url}/" target="_blank">'
    '<strong>Open WasteLens</strong></a>'
))

Your WasteLens Demo:
https://8000-m-s-use4a1-79n0q9ia8prk-a.us-east4.prod.r.colab.dev
